
# 00 — Dataset Inventory and Integrity Audit

## 1. Tujuan

Notebook ini bertujuan untuk melakukan inventarisasi dan pemeriksaan integritas awal terhadap dataset GNSS RINEX yang digunakan dalam penelitian.

Pemeriksaan difokuskan pada identitas data, metadata header, rentang waktu observasi, interval epoch, serta kesesuaian tanggal observasi dengan target penelitian.

## 2. Dataset

- **Stasiun utama:** BAKO
- **Lokasi:** Indonesia
- **Peristiwa:** Halloween Geomagnetic Storm 2003
- **Periode target:** 27 Oktober–3 November 2003
- **DOY target:** 300–307
- **Format data:** RINEX 2.10
- **Jenis observasi:** GPS

## 3. Ruang Lingkup

1. Inventarisasi file RINEX.
2. Pemeriksaan identitas dan metadata header.
3. Pemeriksaan tanggal dan waktu observasi.
4. Pemeriksaan jumlah epoch dan interval waktu.
5. Identifikasi ketidaksesuaian serta kelayakan dataset.

## 4. Batasan

Notebook ini tidak melakukan parsing seluruh observasi satelit, quality control, deteksi cycle slip, maupun perhitungan STEC.

## 5. Output

- Dataset inventory.
- Header metadata summary.
- Epoch coverage summary.
- Dataset integrity audit report.

In [1]:
# CELL 1 — IMPORT LIBRARIES

from pathlib import Path
import os
import re
import pandas as pd
from datetime import datetime

print("Library berhasil diimpor.")
print("Pandas version:", pd.__version__)

Library berhasil diimpor.
Pandas version: 3.0.2


In [2]:
# CELL 2 — WORKSPACE DIRECTORY

workspace_dir = Path.cwd().parent

print("Workspace utama:")
print(workspace_dir)

print("\nPemeriksaan struktur direktori:")

required_folders = [
    "data",
    "documentations",
    "notebooks",
    "Output"
]

for folder in required_folders:
    folder_path = workspace_dir / folder

    status = "Tersedia" if folder_path.is_dir() else "Belum tersedia"

    print(f"{folder:<20}: {status}")

Workspace utama:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final

Pemeriksaan struktur direktori:
data                : Tersedia
documentations      : Tersedia
notebooks           : Tersedia
Output              : Tersedia


In [3]:
# CELL 3 — DEFINE PROJECT DIRECTORIES

data_dir = workspace_dir / "data"
raw_data_dir = data_dir / "raw"
external_data_dir = data_dir / "external"

notebook_dir = workspace_dir / "notebooks"
output_dir = workspace_dir / "Output"
documentation_dir = workspace_dir / "documentations"

print("PROJECT DIRECTORY CONFIGURATION")
print("=" * 45)

directories = {
    "Workspace": workspace_dir,
    "Raw Data": raw_data_dir,
    "External Data": external_data_dir,
    "Notebooks": notebook_dir,
    "Output": output_dir,
    "Documentation": documentation_dir
}

for name, path in directories.items():
    print(f"{name:<20}: {path}")
    print(f"{'':<20}  Exists: {path.exists()}")

PROJECT DIRECTORY CONFIGURATION
Workspace           : c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final
                      Exists: True
Raw Data            : c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\data\raw
                      Exists: True
External Data       : c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\data\external
                      Exists: True
Notebooks           : c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\notebooks
                      Exists: True
Output              : c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output
                      Exists: True
Documentation       : c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\documentations
                      Exists: True


In [4]:
# CELL 4 — RAW DATA FILE INVENTORY

raw_files = sorted([
    file for file in raw_data_dir.rglob("*")
    if file.is_file()
])

print("RAW DATA INVENTORY")
print("=" * 60)
print(f"Lokasi folder : {raw_data_dir}")
print(f"Jumlah file   : {len(raw_files)}")
print("=" * 60)

for i, file in enumerate(raw_files, start=1):
    relative_path = file.relative_to(raw_data_dir)
    file_size = file.stat().st_size

    print(
        f"{i:03d}. {relative_path} "
        f"| Size: {file_size:,} bytes"
    )

RAW DATA INVENTORY
Lokasi folder : c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\data\raw
Jumlah file   : 8
001. DOY300\bako3000.03o | Size: 2,103,424 bytes
002. DOY301\bako3010.03o | Size: 2,104,550 bytes
003. DOY302\bako3020.03o | Size: 2,105,203 bytes
004. DOY303\bako3030.03o | Size: 2,104,572 bytes
005. DOY304\bako3040.03o | Size: 2,096,487 bytes
006. DOY305\bako3050.03o | Size: 2,105,270 bytes
007. DOY306\bako3060.03o | Size: 2,104,740 bytes
008. DOY307\bako3070.03o | Size: 2,105,109 bytes


In [5]:
# CELL 5 — CREATE DATASET INVENTORY TABLE

inventory_records = []

for file in raw_files:
    inventory_records.append({
        "Nama_File": file.name,
        "Folder_DOY": file.parent.name,
        "Format": file.suffix,
        "Ukuran_Bytes": file.stat().st_size,
        "Ukuran_MB": round(file.stat().st_size / (1024**2), 3),
        "Lokasi_Relatif": str(file.relative_to(workspace_dir))
    })

df_inventory = pd.DataFrame(inventory_records)

display(df_inventory)

,Nama_File,Folder_DOY,Format,Ukuran_Bytes,Ukuran_MB,Lokasi_Relatif
0,bako3000.03o,DOY300,.03o,2103424,2.006,data\raw\DOY300\bako3000.03o
1,bako3010.03o,DOY301,.03o,2104550,2.007,data\raw\DOY301\bako3010.03o
2,bako3020.03o,DOY302,.03o,2105203,2.008,data\raw\DOY302\bako3020.03o
3,bako3030.03o,DOY303,.03o,2104572,2.007,data\raw\DOY303\bako3030.03o
4,bako3040.03o,DOY304,.03o,2096487,1.999,data\raw\DOY304\bako3040.03o
5,bako3050.03o,DOY305,.03o,2105270,2.008,data\raw\DOY305\bako3050.03o
6,bako3060.03o,DOY306,.03o,2104740,2.007,data\raw\DOY306\bako3060.03o
7,bako3070.03o,DOY307,.03o,2105109,2.008,data\raw\DOY307\bako3070.03o


In [6]:
# CELL 6 — SAVE DATASET INVENTORY

inventory_output = output_dir / "dataset_inventory.csv"

df_inventory.to_csv(
    inventory_output,
    index=False
)

print("Dataset inventory berhasil disimpan.")
print("Lokasi:", inventory_output)
print("Jumlah file:", len(df_inventory))

Dataset inventory berhasil disimpan.
Lokasi: c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\dataset_inventory.csv
Jumlah file: 8


In [7]:
# CELL 7 — READ RINEX HEADER

rinex_file = raw_data_dir / "DOY300" / "bako3000.03o"

header_lines = []

with open(rinex_file, "r", encoding="ascii", errors="replace") as f:
    for line in f:
        header_lines.append(line.rstrip("\n"))

        if "END OF HEADER" in line:
            break

print("RINEX HEADER")
print("=" * 80)

for line in header_lines:
    print(line)

print("=" * 80)
print("Jumlah baris header:", len(header_lines))

RINEX HEADER
     2.10           OBSERVATION DATA    G (GPS)             RINEX VERSION / TYPE
teqc  2002Mar14                         20031028 12:06:32UTCPGM / RUN BY / DATE
MSWin2000|IAx86-PII|wcl386 11.0|MSWin95/98/NT/2000|486/DX+  COMMENT
BIT 2 OF LLI FLAGS DATA COLLECTED UNDER A/S CONDITION       COMMENT
bako                                                        MARKER NAME
23101M002                                                   MARKER NUMBER
eka                 bakosurtanal                            OBSERVER / AGENCY
2491                TRIMBLE 4000SSE     NP 7.19 / SP 3.04   REC # / TYPE / VERS
44148               TRM14532.00     NONE                    ANT # / TYPE
 -1836967.8873  6065615.4384  -716260.3127                  APPROX POSITION XYZ
        1.6160        0.0000        0.0000                  ANTENNA: DELTA H/E/N
     1     1                                                WAVELENGTH FACT L1/2
     6    L1    L2    C1    P2    P1    D1                  # / TYPES O

In [8]:
# CELL 8 — EXTRACT RINEX HEADER METADATA

metadata_records = []

for line in header_lines:
    label = line[60:80].strip() if len(line) >= 60 else ""

    if label:
        metadata_records.append({
            "Parameter": label,
            "Isi_Header": line[:60].strip()
        })

df_header_metadata = pd.DataFrame(metadata_records)

display(df_header_metadata)

,Parameter,Isi_Header
0,RINEX VERSION / TYPE,2.10 OBSERVATION DATA G (GPS)
1,PGM / RUN BY / DATE,teqc 2002Mar14 200310...
2,COMMENT,MSWin2000|IAx86-PII|wcl386 11.0|MSWin95/98/NT/...
3,COMMENT,BIT 2 OF LLI FLAGS DATA COLLECTED UNDER A/S CO...
4,MARKER NAME,bako
5,MARKER NUMBER,23101M002
6,OBSERVER / AGENCY,eka bakosurtanal
7,REC # / TYPE / VERS,2491 TRIMBLE 4000SSE NP 7.1...
8,ANT # / TYPE,44148 TRM14532.00 NONE
9,APPROX POSITION XYZ,-1836967.8873 6065615.4384 -716260.3127


In [9]:
# CELL 9 — CHECK FILE IDENTITY AND EXPECTED DATE

file_name = rinex_file.name
folder_name = rinex_file.parent.name

expected_date = pd.Timestamp("2003-10-27")
header_first_obs = pd.Timestamp(
    year=2003,
    month=10,
    day=27,
    hour=0,
    minute=0,
    second=30
)

file_identity_check = pd.DataFrame([{
    "File_Name": file_name,
    "Folder": folder_name,
    "Expected_DOY": 300,
    "Expected_Date": expected_date.date(),
    "Header_First_Observation": header_first_obs,
    "Folder_Match": folder_name == "DOY300",
    "Filename_Match": file_name.lower() == "bako3000.03o",
    "Header_Date_Match": header_first_obs.date() == expected_date.date()
}])

display(file_identity_check)

,File_Name,Folder,Expected_DOY,Expected_Date,Header_First_Observation,Folder_Match,Filename_Match,Header_Date_Match
0,bako3000.03o,DOY300,300,2003-10-27,2003-10-27 00:00:30,True,True,True


In [10]:
# CELL 10 — CHECK OBSERVATION EPOCH COVERAGE AND INTERVAL

rinex_lines = rinex_file.read_text(
    encoding="ascii",
    errors="replace"
).splitlines()

epoch_records = []

for line in rinex_lines:

    if len(line) < 32:
        continue

    try:
        yy = int(line[1:3])
        month = int(line[4:6])
        day = int(line[7:9])
        hour = int(line[10:12])
        minute = int(line[13:15])
        second = float(line[15:26])

        # Target file: observations from 2003
        if yy != 3:
            continue

        if not (
            1 <= month <= 12
            and 1 <= day <= 31
            and 0 <= hour <= 23
            and 0 <= minute <= 59
            and 0 <= second < 61
        ):
            continue

        waktu = pd.Timestamp(
            year=2000 + yy,
            month=month,
            day=day,
            hour=hour,
            minute=minute
        ) + pd.Timedelta(seconds=second)

        flag = int(line[28:29])
        jumlah_sat = int(line[29:32])

        epoch_records.append({
            "Waktu": waktu,
            "Flag": flag,
            "Jumlah_Satelit": jumlah_sat
        })

    except (ValueError, TypeError):
        continue

df_epochs = pd.DataFrame(epoch_records)

df_epochs = df_epochs.sort_values(
    "Waktu"
).reset_index(drop=True)

intervals = df_epochs["Waktu"].diff().dropna()

epoch_summary = pd.DataFrame([{
    "Epoch_Pertama_GPS": df_epochs["Waktu"].iloc[0],
    "Epoch_Terakhir_GPS": df_epochs["Waktu"].iloc[-1],
    "Jumlah_Epoch": len(df_epochs),
    "Interval_Median_Detik": intervals.dt.total_seconds().median(),
    "Interval_Minimum_Detik": intervals.dt.total_seconds().min(),
    "Interval_Maksimum_Detik": intervals.dt.total_seconds().max(),
    "Jumlah_Interval_Tidak_30_Detik": (
        intervals.dt.total_seconds() != 30
    ).sum()
}])

display(epoch_summary)

,Epoch_Pertama_GPS,Epoch_Terakhir_GPS,Jumlah_Epoch,Interval_Median_Detik,Interval_Minimum_Detik,Interval_Maksimum_Detik,Jumlah_Interval_Tidak_30_Detik
0,2003-10-27 00:00:30,2003-10-27 23:59:00.128,2878,30.0,30.0,30.001,128


In [11]:
# CELL 11 — INTERVAL DISTRIBUTION ANALYSIS

interval_seconds = (
    df_epochs["Waktu"]
    .diff()
    .dt.total_seconds()
    .dropna()
)

df_interval_distribution = (
    interval_seconds
    .value_counts()
    .sort_index()
    .reset_index()
)

df_interval_distribution.columns = [
    "Interval_Detik",
    "Jumlah_Kejadian"
]

df_interval_distribution["Persentase"] = (
    df_interval_distribution["Jumlah_Kejadian"]
    / len(interval_seconds) * 100
)

display(df_interval_distribution)

,Interval_Detik,Jumlah_Kejadian,Persentase
0,30.000,2749,95.550921
1,30.001,128,4.449079


In [12]:
# CELL 12 — CHECK EPOCH ORDER, DUPLICATES, AND GAPS

epoch_intervals = (
    df_epochs["Waktu"]
    .diff()
    .dt.total_seconds()
)

epoch_order_check = pd.DataFrame([{
    "Jumlah_Epoch": len(df_epochs),
    "Jumlah_Waktu_Duplikat": df_epochs["Waktu"].duplicated().sum(),
    "Jumlah_Interval_Nol_atau_Negatif": (
        epoch_intervals.dropna() <= 0
    ).sum(),
    "Jumlah_Interval_Lebih_dari_30_001_ms": (
        epoch_intervals.dropna() > 30.001
    ).sum(),
    "Epoch_Terurut_Menaik": df_epochs["Waktu"].is_monotonic_increasing,
    "Jumlah_Tanggal_Unik": df_epochs["Waktu"].dt.date.nunique()
}])

display(epoch_order_check)

,Jumlah_Epoch,Jumlah_Waktu_Duplikat,Jumlah_Interval_Nol_atau_Negatif,Jumlah_Interval_Lebih_dari_30_001_ms,Epoch_Terurut_Menaik,Jumlah_Tanggal_Unik
0,2878,0,0,0,True,1


In [13]:
# CELL 13 — SATELLITE COUNT DISTRIBUTION PER EPOCH

df_satellite_distribution = (
    df_epochs["Jumlah_Satelit"]
    .value_counts()
    .sort_index()
    .reset_index()
)

df_satellite_distribution.columns = [
    "Jumlah_Satelit_Per_Epoch",
    "Frekuensi_Epoch"
]

df_satellite_distribution["Persentase"] = (
    df_satellite_distribution["Frekuensi_Epoch"]
    / len(df_epochs) * 100
)

display(df_satellite_distribution)

print("Jumlah satelit minimum:", df_epochs["Jumlah_Satelit"].min())
print("Jumlah satelit maksimum:", df_epochs["Jumlah_Satelit"].max())
print("Rata-rata satelit per epoch:", df_epochs["Jumlah_Satelit"].mean())

,Jumlah_Satelit_Per_Epoch,Frekuensi_Epoch,Persentase
0,7,177,6.150104
1,8,1294,44.961779
2,9,1407,48.888117


Jumlah satelit minimum: 7
Jumlah satelit maksimum: 9
Rata-rata satelit per epoch: 8.427380125086867


In [14]:
# CELL 14 — CHECK SATELLITE LIST CONSISTENCY

satellite_count_check = []

for i, line in enumerate(rinex_lines):

    if len(line) < 32:
        continue

    try:
        yy = int(line[1:3])
        month = int(line[4:6])
        day = int(line[7:9])
        hour = int(line[10:12])
        minute = int(line[13:15])
        second = float(line[15:26])

        if yy != 3:
            continue

        if not (
            1 <= month <= 12
            and 1 <= day <= 31
            and 0 <= hour <= 23
            and 0 <= minute <= 59
            and 0 <= second < 61
        ):
            continue

        jumlah_sat = int(line[29:32])
        sat_text = line[32:68]

        satellite_ids = [
            sat_text[j:j+3].strip()
            for j in range(0, len(sat_text), 3)
            if sat_text[j:j+3].strip()
        ]

        satellite_count_check.append({
            "Jumlah_Satelit_Header_Epoch": jumlah_sat,
            "Jumlah_ID_Terbaca": len(satellite_ids),
            "Sesuai": jumlah_sat == len(satellite_ids),
            "Daftar_Satelit": satellite_ids
        })

    except (ValueError, TypeError):
        continue

df_satellite_count_check = pd.DataFrame(
    satellite_count_check
)

display(
    df_satellite_count_check[
        df_satellite_count_check["Sesuai"] == False
    ]
)

print(
    "Jumlah epoch yang diperiksa:",
    len(df_satellite_count_check)
)

print(
    "Jumlah ketidaksesuaian:",
    (~df_satellite_count_check["Sesuai"]).sum()
)

,Jumlah_Satelit_Header_Epoch,Jumlah_ID_Terbaca,Sesuai,Daftar_Satelit


Jumlah epoch yang diperiksa: 2878
Jumlah ketidaksesuaian: 0


In [15]:
# CELL 15 — EPOCH FLAG DISTRIBUTION

df_epoch_flag = (
    df_epochs["Flag"]
    .value_counts()
    .sort_index()
    .reset_index()
)

df_epoch_flag.columns = [
    "Epoch_Flag",
    "Jumlah_Record"
]

flag_description = {
    0: "Observasi normal",
    1: "Power failure antara epoch sebelumnya dan epoch ini",
    2: "Event khusus",
    3: "Event khusus",
    4: "Event khusus",
    5: "Event khusus",
    6: "Cycle slip records"
}

df_epoch_flag["Keterangan"] = (
    df_epoch_flag["Epoch_Flag"]
    .map(flag_description)
    .fillna("Flag tidak dikenal")
)

display(df_epoch_flag)

print("Total record:", len(df_epochs))
print(
    "Record dengan flag 0:",
    (df_epochs["Flag"] == 0).sum()
)
print(
    "Record dengan flag bukan 0:",
    (df_epochs["Flag"] != 0).sum()
)

,Epoch_Flag,Jumlah_Record,Keterangan
0,0,2878,Observasi normal


Total record: 2878
Record dengan flag 0: 2878
Record dengan flag bukan 0: 0


In [16]:
# CELL 16 — EXPECTED DAILY EPOCH COVERAGE

expected_epochs = pd.date_range(
    start="2003-10-27 00:00:00",
    end="2003-10-27 23:59:30",
    freq="30s"
)

observed_epochs = pd.DatetimeIndex(df_epochs["Waktu"])

missing_epochs = expected_epochs.difference(observed_epochs)
unexpected_epochs = observed_epochs.difference(expected_epochs)

coverage_summary = pd.DataFrame([{
    "Expected_Epochs": len(expected_epochs),
    "Observed_Epochs": len(observed_epochs),
    "Missing_Epochs": len(missing_epochs),
    "Unexpected_Epochs": len(unexpected_epochs),
    "Coverage_Percent": (
        len(observed_epochs) / len(expected_epochs) * 100
    )
}])

display(coverage_summary)

print("Contoh epoch yang tidak ditemukan:")
display(missing_epochs[:10].to_frame(name="Missing_Epoch"))

,Expected_Epochs,Observed_Epochs,Missing_Epochs,Unexpected_Epochs,Coverage_Percent
0,2880,2878,2877,2875,99.930556


Contoh epoch yang tidak ditemukan:


,Missing_Epoch
2003-10-27 00:00:00,2003-10-27 00:00:00
2003-10-27 00:02:00,2003-10-27 00:02:00
2003-10-27 00:02:30,2003-10-27 00:02:30
2003-10-27 00:03:00,2003-10-27 00:03:00
2003-10-27 00:03:30,2003-10-27 00:03:30
2003-10-27 00:04:00,2003-10-27 00:04:00
2003-10-27 00:04:30,2003-10-27 00:04:30
2003-10-27 00:05:00,2003-10-27 00:05:00
2003-10-27 00:05:30,2003-10-27 00:05:30
2003-10-27 00:06:00,2003-10-27 00:06:00


In [17]:
# CELL 16 (REVISED) — MAP OBSERVED EPOCHS TO NOMINAL 30-SECOND GRID

import numpy as np

day_start = pd.Timestamp("2003-10-27 00:00:00")

expected_epochs = pd.date_range(
    start=day_start,
    end="2003-10-27 23:59:30",
    freq="30s"
)

observed_epochs = pd.DatetimeIndex(df_epochs["Waktu"])

# Hitung posisi setiap epoch terhadap awal hari dalam satuan detik
elapsed_seconds = (
    observed_epochs - day_start
).total_seconds()

# Petakan ke slot grid 30 detik terdekat
nearest_slot = np.rint(
    elapsed_seconds / 30
).astype(int)

mapped_epochs = day_start + pd.to_timedelta(
    nearest_slot * 30,
    unit="s"
)

# Hitung deviasi timestamp aktual terhadap grid nominal
residual_seconds = (
    observed_epochs - mapped_epochs
).total_seconds()

mapping_tolerance = 0.5

valid_mapping = (
    (np.abs(residual_seconds) <= mapping_tolerance)
    & (nearest_slot >= 0)
    & (nearest_slot < len(expected_epochs))
)

mapped_valid = mapped_epochs[valid_mapping]

# Periksa slot grid yang terisi lebih dari satu kali
mapped_counts = pd.Series(mapped_valid).value_counts()

duplicate_slots = mapped_counts[
    mapped_counts > 1
]

# Slot grid yang tidak memiliki epoch
missing_epochs = expected_epochs.difference(
    pd.DatetimeIndex(mapped_valid.unique())
)

coverage_summary = pd.DataFrame([{
    "Expected_Epochs": len(expected_epochs),
    "Observed_Epochs": len(observed_epochs),
    "Successfully_Mapped": valid_mapping.sum(),
    "Unmapped_Epochs": (~valid_mapping).sum(),
    "Missing_Grid_Slots": len(missing_epochs),
    "Duplicate_Grid_Slots": len(duplicate_slots),
    "Maximum_Absolute_Residual_ms": (
        np.abs(residual_seconds).max() * 1000
    ),
    "Coverage_Percent": (
        len(mapped_valid.unique())
        / len(expected_epochs) * 100
    )
}])

display(coverage_summary)

print("Grid epoch yang tidak ditemukan:")
display(
    pd.DataFrame({
        "Missing_Epoch": missing_epochs
    })
)

print("Distribusi deviasi timestamp terhadap grid (detik):")
display(
    pd.Series(residual_seconds).describe()
)

,Expected_Epochs,Observed_Epochs,Successfully_Mapped,Unmapped_Epochs,Missing_Grid_Slots,Duplicate_Grid_Slots,Maximum_Absolute_Residual_ms,Coverage_Percent
0,2880,2878,2878,0,2,0,128.0,99.930556


Grid epoch yang tidak ditemukan:


,Missing_Epoch
0,2003-10-27 00:00:00
1,2003-10-27 23:59:30


Distribusi deviasi timestamp terhadap grid (detik):


count    2878.000000
mean        0.064116
std         0.036922
min         0.000000
25%         0.032000
50%         0.064000
75%         0.096000
max         0.128000
Name: Waktu, dtype: float64

In [18]:
# CELL 17 — SAVE DOY300 INTEGRITY AUDIT SUMMARY

audit_summary = pd.DataFrame([{
    "File_Name": rinex_file.name,
    "Folder": rinex_file.parent.name,
    "Expected_DOY": 300,
    "Expected_Date": "2003-10-27",

    "Epoch_First_GPS": df_epochs["Waktu"].iloc[0],
    "Epoch_Last_GPS": df_epochs["Waktu"].iloc[-1],
    "Observed_Epochs": len(df_epochs),

    "Expected_Daily_Epochs": len(expected_epochs),
    "Successfully_Mapped": int(valid_mapping.sum()),
    "Unmapped_Epochs": int((~valid_mapping).sum()),
    "Missing_Grid_Slots": len(missing_epochs),
    "Duplicate_Grid_Slots": len(duplicate_slots),

    "Maximum_Absolute_Residual_ms": (
        np.abs(residual_seconds).max() * 1000
    ),

    "Temporal_Coverage_Percent": (
        len(mapped_valid.unique())
        / len(expected_epochs) * 100
    ),

    "Duplicate_Timestamps": (
        df_epochs["Waktu"].duplicated().sum()
    ),

    "Nonzero_Epoch_Flags": (
        (df_epochs["Flag"] != 0).sum()
    ),

    "Satellite_ID_Count_Mismatches": (
        (~df_satellite_count_check["Sesuai"]).sum()
    )
}])

display(audit_summary)

audit_output_path = output_dir / "doy300_integrity_audit_summary.csv"

audit_summary.to_csv(
    audit_output_path,
    index=False
)

print("Audit summary saved to:")
print(audit_output_path)

,File_Name,Folder,Expected_DOY,Expected_Date,Epoch_First_GPS,Epoch_Last_GPS,Observed_Epochs,Expected_Daily_Epochs,Successfully_Mapped,Unmapped_Epochs,Missing_Grid_Slots,Duplicate_Grid_Slots,Maximum_Absolute_Residual_ms,Temporal_Coverage_Percent,Duplicate_Timestamps,Nonzero_Epoch_Flags,Satellite_ID_Count_Mismatches
0,bako3000.03o,DOY300,300,2003-10-27,2003-10-27 00:00:30,2003-10-27 23:59:00.128,2878,2880,2878,0,2,0,128.0,99.930556,0,0,0


Audit summary saved to:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\doy300_integrity_audit_summary.csv


In [19]:
# CELL 18 — INVENTORY RINEX FILES PER DOY FOLDER

from pathlib import Path
import pandas as pd

# Mengambil lokasi data/raw dari path file yang sudah diperiksa
raw_dir = rinex_file.parent.parent

inventory_doy = []

for doy in range(300, 308):

    doy_folder = raw_dir / f"DOY{doy}"

    if doy_folder.exists():
        rinex_files = [
            f for f in doy_folder.iterdir()
            if f.is_file()
        ]

        if rinex_files:
            for file in rinex_files:
                inventory_doy.append({
                    "DOY_Folder": f"DOY{doy}",
                    "File_Name": file.name,
                    "File_Size_Bytes": file.stat().st_size,
                    "Status": "FILE_FOUND"
                })
        else:
            inventory_doy.append({
                "DOY_Folder": f"DOY{doy}",
                "File_Name": None,
                "File_Size_Bytes": 0,
                "Status": "EMPTY"
            })

    else:
        inventory_doy.append({
            "DOY_Folder": f"DOY{doy}",
            "File_Name": None,
            "File_Size_Bytes": 0,
            "Status": "FOLDER_NOT_FOUND"
        })

df_inventory_doy = pd.DataFrame(inventory_doy)

display(df_inventory_doy)

print("Total file ditemukan:",
      (df_inventory_doy["Status"] == "FILE_FOUND").sum())

print("Total folder kosong:",
      (df_inventory_doy["Status"] == "EMPTY").sum())

print("Total folder tidak ditemukan:",
      (df_inventory_doy["Status"] == "FOLDER_NOT_FOUND").sum())

,DOY_Folder,File_Name,File_Size_Bytes,Status
0,DOY300,bako3000.03o,2103424,FILE_FOUND
1,DOY301,bako3010.03o,2104550,FILE_FOUND
2,DOY302,bako3020.03o,2105203,FILE_FOUND
3,DOY303,bako3030.03o,2104572,FILE_FOUND
4,DOY304,bako3040.03o,2096487,FILE_FOUND
5,DOY305,bako3050.03o,2105270,FILE_FOUND
6,DOY306,bako3060.03o,2104740,FILE_FOUND
7,DOY307,bako3070.03o,2105109,FILE_FOUND


Total file ditemukan: 8
Total folder kosong: 0
Total folder tidak ditemukan: 0


In [20]:
# CELL 19 — HEADER DATE AND DATASET IDENTITY AUDIT

import re
from datetime import datetime, timedelta
import pandas as pd

header_audit = []

for doy in range(300, 308):

    doy_folder = raw_dir / f"DOY{doy}"

    for file in sorted(doy_folder.iterdir()):

        if not file.is_file():
            continue

        try:
            with open(file, "r", errors="replace") as f:

                first_obs = None
                header_complete = False

                for line in f:

                    label = line[60:80].strip()

                    if label == "TIME OF FIRST OBS":
                        tokens = line[:60].split()

                        first_obs = datetime(
                            int(tokens[0]),
                            int(tokens[1]),
                            int(tokens[2]),
                            int(tokens[3]),
                            int(tokens[4])
                        ) + timedelta(seconds=float(tokens[5]))

                    if label == "END OF HEADER":
                        header_complete = True
                        break

            # DOY dari nama file, contoh bako3000.03o
            match = re.search(r"(\d{3})", file.stem)
            filename_doy = int(match.group(1)) if match else None

            expected_date = (
                datetime(2003, 1, 1)
                + timedelta(days=doy - 1)
            ).date()

            header_date = (
                first_obs.date()
                if first_obs is not None
                else None
            )

            folder_match = (
                filename_doy == doy
            )

            header_match = (
                header_date == expected_date
            )

            if not header_complete or first_obs is None:
                status = "HEADER_INCOMPLETE"

            elif folder_match and header_match:
                status = "MATCH"

            else:
                status = "DATE_MISMATCH"

            header_audit.append({
                "DOY_Folder": f"DOY{doy}",
                "File_Name": file.name,
                "Filename_DOY": filename_doy,
                "Expected_Date": expected_date,
                "Header_First_Obs": first_obs,
                "Header_Date": header_date,
                "Folder_Filename_Match": folder_match,
                "Header_Date_Match": header_match,
                "Header_Complete": header_complete,
                "Audit_Status": status
            })

        except Exception as e:

            header_audit.append({
                "DOY_Folder": f"DOY{doy}",
                "File_Name": file.name,
                "Filename_DOY": None,
                "Expected_Date": None,
                "Header_First_Obs": None,
                "Header_Date": None,
                "Folder_Filename_Match": False,
                "Header_Date_Match": False,
                "Header_Complete": False,
                "Audit_Status": f"ERROR: {e}"
            })

df_header_audit = pd.DataFrame(header_audit)

display(df_header_audit)

print("\nRingkasan status audit:")
display(
    df_header_audit["Audit_Status"].value_counts()
)

# Simpan hasil
header_audit_path = output_dir / "dataset_header_identity_audit.csv"

df_header_audit.to_csv(
    header_audit_path,
    index=False
)

print("\nHasil audit disimpan di:")
print(header_audit_path)

,DOY_Folder,File_Name,Filename_DOY,Expected_Date,Header_First_Obs,Header_Date,Folder_Filename_Match,Header_Date_Match,Header_Complete,Audit_Status
0,DOY300,bako3000.03o,300,2003-10-27,2003-10-27 00:00:30,2003-10-27,True,True,True,MATCH
1,DOY301,bako3010.03o,301,2003-10-28,2003-10-28 00:00:30,2003-10-28,True,True,True,MATCH
2,DOY302,bako3020.03o,302,2003-10-29,2003-10-29 00:00:30,2003-10-29,True,True,True,MATCH
3,DOY303,bako3030.03o,303,2003-10-30,2003-10-30 00:00:30,2003-10-30,True,True,True,MATCH
4,DOY304,bako3040.03o,304,2003-10-31,2003-10-31 00:06:00,2003-10-31,True,True,True,MATCH
5,DOY305,bako3050.03o,305,2003-11-01,2003-11-08 00:00:30,2003-11-08,True,False,True,DATE_MISMATCH
6,DOY306,bako3060.03o,306,2003-11-02,2003-11-02 00:00:30,2003-11-02,True,True,True,MATCH
7,DOY307,bako3070.03o,307,2003-11-03,2003-11-03 00:00:30,2003-11-03,True,True,True,MATCH



Ringkasan status audit:


Audit_Status
MATCH            7
DATE_MISMATCH    1
Name: count, dtype: int64


Hasil audit disimpan di:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\dataset_header_identity_audit.csv


In [21]:
# CELL 20 — TEMPORAL AUDIT FOR ALL RINEX FILES

import pandas as pd
from datetime import datetime

temporal_audit = []

for doy in range(300, 308):

    doy_folder = raw_dir / f"DOY{doy}"

    for file in sorted(doy_folder.iterdir()):

        if not file.is_file():
            continue

        epochs = []
        epoch_flags = []

        with open(file, "r", errors="replace") as f:

            # Lewati header
            for line in f:
                if line[60:80].strip() == "END OF HEADER":
                    break

            # Baca epoch records
            for line in f:

                if len(line) < 32 or line[0] != " ":
                    continue

                try:
                    yy = int(line[1:3])
                    month = int(line[4:6])
                    day = int(line[7:9])
                    hour = int(line[10:12])
                    minute = int(line[13:15])
                    second = float(line[15:26])

                    flag = int(line[28:29])

                    if not (
                        1 <= month <= 12
                        and 1 <= day <= 31
                        and 0 <= hour <= 23
                        and 0 <= minute <= 59
                        and 0 <= second < 61
                    ):
                        continue

                    waktu = (
                        pd.Timestamp(
                            year=2000 + yy,
                            month=month,
                            day=day,
                            hour=hour,
                            minute=minute
                        )
                        + pd.Timedelta(seconds=second)
                    )

                    epochs.append(waktu)
                    epoch_flags.append(flag)

                except (ValueError, TypeError):
                    continue

        epoch_index = pd.DatetimeIndex(epochs)

        if len(epoch_index) > 1:

            intervals = (
                epoch_index.to_series()
                .diff()
                .dropna()
                .dt.total_seconds()
            )

            median_interval = intervals.median()
            min_interval = intervals.min()
            max_interval = intervals.max()

        else:
            median_interval = None
            min_interval = None
            max_interval = None

        expected_date = (
            pd.Timestamp("2003-01-01")
            + pd.Timedelta(days=doy - 1)
        ).date()

        temporal_audit.append({
            "DOY_Folder": f"DOY{doy}",
            "File_Name": file.name,
            "Expected_Date": expected_date,
            "First_Epoch_GPS": (
                epoch_index[0] if len(epoch_index) else None
            ),
            "Last_Epoch_GPS": (
                epoch_index[-1] if len(epoch_index) else None
            ),
            "Observed_Epochs": len(epoch_index),
            "Median_Interval_s": median_interval,
            "Minimum_Interval_s": min_interval,
            "Maximum_Interval_s": max_interval,
            "Nonzero_Epoch_Flags": sum(
                flag != 0 for flag in epoch_flags
            ),
            "Unique_Observation_Dates": (
                epoch_index.normalize().nunique()
                if len(epoch_index) else 0
            )
        })

df_temporal_audit = pd.DataFrame(temporal_audit)

display(df_temporal_audit)

# Simpan hasil
temporal_audit_path = (
    output_dir / "dataset_temporal_audit.csv"
)

df_temporal_audit.to_csv(
    temporal_audit_path,
    index=False
)

print("Temporal audit saved to:")
print(temporal_audit_path)

,DOY_Folder,File_Name,Expected_Date,First_Epoch_GPS,Last_Epoch_GPS,Observed_Epochs,Median_Interval_s,Minimum_Interval_s,Maximum_Interval_s,Nonzero_Epoch_Flags,Unique_Observation_Dates
0,DOY300,bako3000.03o,2003-10-27,2003-10-27 00:00:30,2003-10-27 23:59:00.128,2878,30.0,30.0,30.001,0,1
1,DOY301,bako3010.03o,2003-10-28,2003-10-28 00:00:30,2003-10-28 23:59:00.127,2878,30.0,30.0,30.001,0,1
2,DOY302,bako3020.03o,2003-10-29,2003-10-29 00:00:30,2003-10-29 23:59:00.125,2878,30.0,30.0,30.001,0,1
3,DOY303,bako3030.03o,2003-10-30,2003-10-30 00:00:30,2003-10-30 23:59:00.124,2878,30.0,30.0,30.001,0,1
4,DOY304,bako3040.03o,2003-10-31,2003-10-31 00:06:00,2003-10-31 23:59:00.125,2867,30.0,30.0,30.001,0,1
5,DOY305,bako3050.03o,2003-11-01,2003-11-08 00:00:30,2003-11-08 23:59:00.127,2878,30.0,30.0,30.001,0,1
6,DOY306,bako3060.03o,2003-11-02,2003-11-02 00:00:30,2003-11-02 23:59:00.129,2878,30.0,30.0,30.001,0,1
7,DOY307,bako3070.03o,2003-11-03,2003-11-03 00:00:30,2003-11-03 23:59:00.128,2878,30.0,30.0,30.001,0,1


Temporal audit saved to:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\dataset_temporal_audit.csv


In [22]:
# CELL 21 — DAILY GRID COVERAGE AUDIT FOR ALL FILES

import numpy as np
import pandas as pd

grid_audit = []

for doy in range(300, 308):

    doy_folder = raw_dir / f"DOY{doy}"

    for file in sorted(doy_folder.iterdir()):

        if not file.is_file():
            continue

        epochs = []

        with open(file, "r", errors="replace") as f:

            for line in f:
                if line[60:80].strip() == "END OF HEADER":
                    break

            for line in f:

                if len(line) < 32 or line[0] != " ":
                    continue

                try:
                    yy = int(line[1:3])
                    month = int(line[4:6])
                    day = int(line[7:9])
                    hour = int(line[10:12])
                    minute = int(line[13:15])
                    second = float(line[15:26])

                    if not (
                        1 <= month <= 12
                        and 1 <= day <= 31
                        and 0 <= hour <= 23
                        and 0 <= minute <= 59
                        and 0 <= second < 61
                    ):
                        continue

                    waktu = (
                        pd.Timestamp(
                            year=2000 + yy,
                            month=month,
                            day=day,
                            hour=hour,
                            minute=minute
                        )
                        + pd.Timedelta(seconds=second)
                    )

                    epochs.append(waktu)

                except (ValueError, TypeError):
                    continue

        observed = pd.DatetimeIndex(epochs)

        if len(observed) == 0:
            continue

        # Grid menggunakan tanggal observasi aktual
        actual_date = observed[0].normalize()

        expected_grid = pd.date_range(
            start=actual_date,
            periods=2880,
            freq="30s"
        )

        elapsed = (
            observed - actual_date
        ).total_seconds()

        nearest_slot = np.rint(
            elapsed / 30
        ).astype(int)

        mapped = (
            actual_date
            + pd.to_timedelta(nearest_slot * 30, unit="s")
        )

        residual = (
            observed - mapped
        ).total_seconds()

        valid = (
            (np.abs(residual) <= 0.5)
            & (nearest_slot >= 0)
            & (nearest_slot < len(expected_grid))
        )

        mapped_valid = mapped[valid]

        counts = pd.Series(mapped_valid).value_counts()

        duplicate_slots = counts[counts > 1]

        missing_slots = expected_grid.difference(
            pd.DatetimeIndex(mapped_valid.unique())
        )

        internal_missing = [
            t for t in missing_slots
            if actual_date < t < expected_grid[-1]
        ]

        grid_audit.append({
            "DOY_Folder": f"DOY{doy}",
            "File_Name": file.name,
            "Actual_Observation_Date": actual_date.date(),
            "Expected_Epochs": len(expected_grid),
            "Observed_Epochs": len(observed),
            "Successfully_Mapped": int(valid.sum()),
            "Unmapped_Epochs": int((~valid).sum()),
            "Missing_Grid_Slots": len(missing_slots),
            "Internal_Missing_Slots": len(internal_missing),
            "Duplicate_Grid_Slots": len(duplicate_slots),
            "Maximum_Residual_ms": (
                np.abs(residual).max() * 1000
            ),
            "Coverage_Percent": (
                len(mapped_valid.unique())
                / len(expected_grid) * 100
            )
        })

df_grid_audit = pd.DataFrame(grid_audit)

display(df_grid_audit)

grid_audit_path = output_dir / "dataset_grid_coverage_audit.csv"

df_grid_audit.to_csv(
    grid_audit_path,
    index=False
)

print("Grid coverage audit saved to:")
print(grid_audit_path)

,DOY_Folder,File_Name,Actual_Observation_Date,Expected_Epochs,Observed_Epochs,Successfully_Mapped,Unmapped_Epochs,Missing_Grid_Slots,Internal_Missing_Slots,Duplicate_Grid_Slots,Maximum_Residual_ms,Coverage_Percent
0,DOY300,bako3000.03o,2003-10-27,2880,2878,2878,0,2,0,0,128.0,99.930556
1,DOY301,bako3010.03o,2003-10-28,2880,2878,2878,0,2,0,0,127.0,99.930556
2,DOY302,bako3020.03o,2003-10-29,2880,2878,2878,0,2,0,0,125.0,99.930556
3,DOY303,bako3030.03o,2003-10-30,2880,2878,2878,0,2,0,0,124.0,99.930556
4,DOY304,bako3040.03o,2003-10-31,2880,2867,2867,0,13,11,0,125.0,99.548611
5,DOY305,bako3050.03o,2003-11-08,2880,2878,2878,0,2,0,0,127.0,99.930556
6,DOY306,bako3060.03o,2003-11-02,2880,2878,2878,0,2,0,0,129.0,99.930556
7,DOY307,bako3070.03o,2003-11-03,2880,2878,2878,0,2,0,0,128.0,99.930556


Grid coverage audit saved to:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\dataset_grid_coverage_audit.csv


In [23]:
# CELL 22 — INITIAL DATASET ELIGIBILITY CLASSIFICATION

eligibility = df_header_audit[
    [
        "DOY_Folder",
        "File_Name",
        "Audit_Status"
    ]
].merge(
    df_grid_audit[
        [
            "DOY_Folder",
            "Missing_Grid_Slots",
            "Internal_Missing_Slots",
            "Unmapped_Epochs",
            "Coverage_Percent"
        ]
    ],
    on="DOY_Folder",
    how="left"
)

def classify_file(row):

    if row["Audit_Status"] != "MATCH":
        return "HOLD_IDENTITY_MISMATCH"

    if row["Unmapped_Epochs"] > 0:
        return "HOLD_TEMPORAL_MAPPING"

    if row["Internal_Missing_Slots"] > 0:
        return "REVIEW_INTERNAL_GAPS"

    if row["Missing_Grid_Slots"] > 2:
        return "REVIEW_TEMPORAL_COVERAGE"

    return "ELIGIBLE_FOR_NEXT_STAGE"

eligibility["Initial_Status"] = eligibility.apply(
    classify_file,
    axis=1
)

display(eligibility)

eligibility_path = (
    output_dir / "dataset_initial_eligibility.csv"
)

eligibility.to_csv(
    eligibility_path,
    index=False
)

print("Eligibility report saved to:")
print(eligibility_path)

,DOY_Folder,File_Name,Audit_Status,Missing_Grid_Slots,Internal_Missing_Slots,Unmapped_Epochs,Coverage_Percent,Initial_Status
0,DOY300,bako3000.03o,MATCH,2,0,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
1,DOY301,bako3010.03o,MATCH,2,0,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
2,DOY302,bako3020.03o,MATCH,2,0,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
3,DOY303,bako3030.03o,MATCH,2,0,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
4,DOY304,bako3040.03o,MATCH,13,11,0,99.548611,REVIEW_INTERNAL_GAPS
5,DOY305,bako3050.03o,DATE_MISMATCH,2,0,0,99.930556,HOLD_IDENTITY_MISMATCH
6,DOY306,bako3060.03o,MATCH,2,0,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
7,DOY307,bako3070.03o,MATCH,2,0,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE


Eligibility report saved to:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\dataset_initial_eligibility.csv


In [24]:
# CELL 21 REVISED — CLASSIFY MISSING SLOTS CORRECTLY

import numpy as np
import pandas as pd

grid_audit = []

for doy in range(300, 308):

    doy_folder = raw_dir / f"DOY{doy}"

    for file in sorted(doy_folder.iterdir()):

        if not file.is_file():
            continue

        epochs = []

        with open(file, "r", errors="replace") as f:

            for line in f:
                if line[60:80].strip() == "END OF HEADER":
                    break

            for line in f:

                if len(line) < 32 or line[0] != " ":
                    continue

                try:
                    yy = int(line[1:3])
                    month = int(line[4:6])
                    day = int(line[7:9])
                    hour = int(line[10:12])
                    minute = int(line[13:15])
                    second = float(line[15:26])

                    waktu = (
                        pd.Timestamp(
                            year=2000 + yy,
                            month=month,
                            day=day,
                            hour=hour,
                            minute=minute
                        )
                        + pd.Timedelta(seconds=second)
                    )

                    epochs.append(waktu)

                except (ValueError, TypeError):
                    continue

        observed = pd.DatetimeIndex(epochs)

        if len(observed) == 0:
            continue

        actual_date = observed[0].normalize()

        expected_grid = pd.date_range(
            start=actual_date,
            periods=2880,
            freq="30s"
        )

        elapsed = (observed - actual_date).total_seconds()

        nearest_slot = np.rint(elapsed / 30).astype(int)

        mapped = (
            actual_date
            + pd.to_timedelta(nearest_slot * 30, unit="s")
        )

        residual = (observed - mapped).total_seconds()

        valid = (
            (np.abs(residual) <= 0.5)
            & (nearest_slot >= 0)
            & (nearest_slot < len(expected_grid))
        )

        mapped_valid = mapped[valid]

        unique_mapped = pd.DatetimeIndex(
            mapped_valid.unique()
        ).sort_values()

        missing_slots = expected_grid.difference(unique_mapped)

        if len(unique_mapped) > 0:

            first_mapped = unique_mapped[0]
            last_mapped = unique_mapped[-1]

            leading_missing = [
                t for t in missing_slots
                if t < first_mapped
            ]

            internal_missing = [
                t for t in missing_slots
                if first_mapped < t < last_mapped
            ]

            trailing_missing = [
                t for t in missing_slots
                if t > last_mapped
            ]

        else:
            leading_missing = []
            internal_missing = []
            trailing_missing = list(missing_slots)

        counts = pd.Series(mapped_valid).value_counts()
        duplicate_slots = counts[counts > 1]

        grid_audit.append({
            "DOY_Folder": f"DOY{doy}",
            "File_Name": file.name,
            "Actual_Observation_Date": actual_date.date(),
            "Expected_Epochs": len(expected_grid),
            "Observed_Epochs": len(observed),
            "Successfully_Mapped": int(valid.sum()),
            "Unmapped_Epochs": int((~valid).sum()),
            "Missing_Grid_Slots": len(missing_slots),
            "Leading_Missing_Slots": len(leading_missing),
            "Internal_Missing_Slots": len(internal_missing),
            "Trailing_Missing_Slots": len(trailing_missing),
            "Duplicate_Grid_Slots": len(duplicate_slots),
            "Maximum_Residual_ms": (
                np.abs(residual).max() * 1000
            ),
            "Coverage_Percent": (
                len(unique_mapped) / len(expected_grid) * 100
            )
        })

df_grid_audit = pd.DataFrame(grid_audit)

display(df_grid_audit)

grid_audit_path = output_dir / "dataset_grid_coverage_audit.csv"

df_grid_audit.to_csv(grid_audit_path, index=False)

print("Revised grid coverage audit saved to:")
print(grid_audit_path)

,DOY_Folder,File_Name,Actual_Observation_Date,Expected_Epochs,Observed_Epochs,Successfully_Mapped,Unmapped_Epochs,Missing_Grid_Slots,Leading_Missing_Slots,Internal_Missing_Slots,Trailing_Missing_Slots,Duplicate_Grid_Slots,Maximum_Residual_ms,Coverage_Percent
0,DOY300,bako3000.03o,2003-10-27,2880,2878,2878,0,2,1,0,1,0,128.0,99.930556
1,DOY301,bako3010.03o,2003-10-28,2880,2878,2878,0,2,1,0,1,0,127.0,99.930556
2,DOY302,bako3020.03o,2003-10-29,2880,2878,2878,0,2,1,0,1,0,125.0,99.930556
3,DOY303,bako3030.03o,2003-10-30,2880,2878,2878,0,2,1,0,1,0,124.0,99.930556
4,DOY304,bako3040.03o,2003-10-31,2880,2867,2867,0,13,12,0,1,0,125.0,99.548611
5,DOY305,bako3050.03o,2003-11-08,2880,2878,2878,0,2,1,0,1,0,127.0,99.930556
6,DOY306,bako3060.03o,2003-11-02,2880,2878,2878,0,2,1,0,1,0,129.0,99.930556
7,DOY307,bako3070.03o,2003-11-03,2880,2878,2878,0,2,1,0,1,0,128.0,99.930556


Revised grid coverage audit saved to:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\dataset_grid_coverage_audit.csv


In [25]:
# CELL 22 REVISED — INITIAL DATASET ELIGIBILITY

eligibility = df_header_audit[
    [
        "DOY_Folder",
        "File_Name",
        "Audit_Status"
    ]
].merge(
    df_grid_audit[
        [
            "DOY_Folder",
            "Actual_Observation_Date",
            "Leading_Missing_Slots",
            "Internal_Missing_Slots",
            "Trailing_Missing_Slots",
            "Unmapped_Epochs",
            "Coverage_Percent"
        ]
    ],
    on="DOY_Folder",
    how="left"
)

def classify_file(row):

    if row["Audit_Status"] != "MATCH":
        return "HOLD_IDENTITY_MISMATCH"

    if row["Unmapped_Epochs"] > 0:
        return "HOLD_TEMPORAL_MAPPING"

    if row["Internal_Missing_Slots"] > 0:
        return "REVIEW_INTERNAL_GAPS"

    boundary_missing = (
        row["Leading_Missing_Slots"]
        + row["Trailing_Missing_Slots"]
    )

    if boundary_missing > 2:
        return "REVIEW_BOUNDARY_COVERAGE"

    return "ELIGIBLE_FOR_NEXT_STAGE"


eligibility["Initial_Status"] = eligibility.apply(
    classify_file,
    axis=1
)

display(eligibility)

eligibility_path = (
    output_dir / "dataset_initial_eligibility.csv"
)

eligibility.to_csv(
    eligibility_path,
    index=False
)

print("Eligibility report saved to:")
print(eligibility_path)

,DOY_Folder,File_Name,Audit_Status,Actual_Observation_Date,Leading_Missing_Slots,Internal_Missing_Slots,Trailing_Missing_Slots,Unmapped_Epochs,Coverage_Percent,Initial_Status
0,DOY300,bako3000.03o,MATCH,2003-10-27,1,0,1,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
1,DOY301,bako3010.03o,MATCH,2003-10-28,1,0,1,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
2,DOY302,bako3020.03o,MATCH,2003-10-29,1,0,1,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
3,DOY303,bako3030.03o,MATCH,2003-10-30,1,0,1,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
4,DOY304,bako3040.03o,MATCH,2003-10-31,12,0,1,0,99.548611,REVIEW_BOUNDARY_COVERAGE
5,DOY305,bako3050.03o,DATE_MISMATCH,2003-11-08,1,0,1,0,99.930556,HOLD_IDENTITY_MISMATCH
6,DOY306,bako3060.03o,MATCH,2003-11-02,1,0,1,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
7,DOY307,bako3070.03o,MATCH,2003-11-03,1,0,1,0,99.930556,ELIGIBLE_FOR_NEXT_STAGE


Eligibility report saved to:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\dataset_initial_eligibility.csv


In [26]:
# CELL 23 — FINAL DATASET AUDIT REPORT

# Gabungkan klasifikasi awal dengan detail temporal grid
df_final_audit = eligibility.merge(
    df_grid_audit[
        [
            "DOY_Folder",
            "File_Name",
            "Expected_Epochs",
            "Observed_Epochs",
            "Successfully_Mapped",
            "Missing_Grid_Slots",
            "Duplicate_Grid_Slots",
            "Maximum_Residual_ms"
        ]
    ],
    on=["DOY_Folder", "File_Name"],
    how="left"
)

# Susun urutan kolom laporan
final_columns = [
    "DOY_Folder",
    "File_Name",
    "Audit_Status",
    "Actual_Observation_Date",
    "Expected_Epochs",
    "Observed_Epochs",
    "Successfully_Mapped",
    "Missing_Grid_Slots",
    "Leading_Missing_Slots",
    "Internal_Missing_Slots",
    "Trailing_Missing_Slots",
    "Unmapped_Epochs",
    "Duplicate_Grid_Slots",
    "Maximum_Residual_ms",
    "Coverage_Percent",
    "Initial_Status"
]

df_final_audit = df_final_audit[final_columns]

# Urutkan berdasarkan DOY
df_final_audit = (
    df_final_audit
    .sort_values("DOY_Folder")
    .reset_index(drop=True)
)

display(df_final_audit)

# Simpan laporan CSV
final_csv_path = output_dir / "dataset_final_audit_report.csv"

df_final_audit.to_csv(
    final_csv_path,
    index=False
)

# Buat ringkasan hasil audit
status_counts = (
    df_final_audit["Initial_Status"]
    .value_counts()
)

report_lines = [
    "FINAL DATASET AUDIT REPORT",
    "GNSS-TEC HALLOWEEN STORM 2003",
    "Station: BAKO",
    "",
    f"Total files audited: {len(df_final_audit)}",
    "",
    "INITIAL ELIGIBILITY SUMMARY:"
]

for status, count in status_counts.items():
    report_lines.append(f"- {status}: {count}")

report_lines.extend([
    "",
    "IMPORTANT FINDINGS:",
    "- DOY305 has an observation date mismatch.",
    "- DOY304 has 12 leading and 1 trailing missing grid slots.",
    "- No internal missing grid slots were identified.",
    "- DOY300-303 and DOY306-307 have 1 leading and 1 trailing missing slot each.",
    "",
    "SCOPE AND LIMITATION:",
    "This report documents initial file identity and temporal coverage.",
    "It does not establish observation-level quality or scientific suitability.",
    "Further RINEX parser validation and GNSS observation quality assessment are required."
])

report_text = "\n".join(report_lines)

final_txt_path = output_dir / "dataset_final_audit_report.txt"

with open(final_txt_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("Final audit report saved:")
print(final_csv_path)
print(final_txt_path)

print("\nEligibility summary:")
print(status_counts)

,DOY_Folder,File_Name,Audit_Status,Actual_Observation_Date,Expected_Epochs,Observed_Epochs,Successfully_Mapped,Missing_Grid_Slots,Leading_Missing_Slots,Internal_Missing_Slots,Trailing_Missing_Slots,Unmapped_Epochs,Duplicate_Grid_Slots,Maximum_Residual_ms,Coverage_Percent,Initial_Status
0,DOY300,bako3000.03o,MATCH,2003-10-27,2880,2878,2878,2,1,0,1,0,0,128.0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
1,DOY301,bako3010.03o,MATCH,2003-10-28,2880,2878,2878,2,1,0,1,0,0,127.0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
2,DOY302,bako3020.03o,MATCH,2003-10-29,2880,2878,2878,2,1,0,1,0,0,125.0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
3,DOY303,bako3030.03o,MATCH,2003-10-30,2880,2878,2878,2,1,0,1,0,0,124.0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
4,DOY304,bako3040.03o,MATCH,2003-10-31,2880,2867,2867,13,12,0,1,0,0,125.0,99.548611,REVIEW_BOUNDARY_COVERAGE
5,DOY305,bako3050.03o,DATE_MISMATCH,2003-11-08,2880,2878,2878,2,1,0,1,0,0,127.0,99.930556,HOLD_IDENTITY_MISMATCH
6,DOY306,bako3060.03o,MATCH,2003-11-02,2880,2878,2878,2,1,0,1,0,0,129.0,99.930556,ELIGIBLE_FOR_NEXT_STAGE
7,DOY307,bako3070.03o,MATCH,2003-11-03,2880,2878,2878,2,1,0,1,0,0,128.0,99.930556,ELIGIBLE_FOR_NEXT_STAGE


Final audit report saved:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\dataset_final_audit_report.csv
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\dataset_final_audit_report.txt

Eligibility summary:
Initial_Status
ELIGIBLE_FOR_NEXT_STAGE     6
REVIEW_BOUNDARY_COVERAGE    1
HOLD_IDENTITY_MISMATCH      1
Name: count, dtype: int64


In [27]:
# ============================================================
# CELL 24 — CREATE README PIPELINE v0.1
# ============================================================

from pathlib import Path

# 1. DEFINE PROJECT DIRECTORY
PROJECT_DIR = Path(
    r"C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final"
)

DOCUMENTATION_DIR = PROJECT_DIR / "documentations"
OUTPUT_DIR = PROJECT_DIR / "Output"

# 2. CHECK REQUIRED DIRECTORIES
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Output directory not found: {OUTPUT_DIR}")

DOCUMENTATION_DIR.mkdir(parents=True, exist_ok=True)

# 3. README CONTENT
README_CONTENT = """# GNSS-TEC Processing Pipeline v0.1

## 1. Research Overview

**Research Title:**

Multi Parameter Observations of Solar Storm Impacts on the Indonesian Equator Ionosphere

**Dataset:** GPS RINEX 2.10 observation data from BAKO station.

**Research Event:** Halloween Storm 2003, covering 27 October–3 November 2003 (DOY 300–307).

**Research Objective:** To investigate the response of equatorial ionospheric TEC to geomagnetic storm activity.

## 2. Pipeline Objectives

This pipeline is designed to process GNSS observation data systematically and reproducibly.

The planned processing stages are:

1. Dataset Inventory and Integrity Audit.
2. RINEX Data Ingestion and Parser Development.
3. RINEX Parser Validation and Data Reconciliation.
4. GNSS Observation Quality Assessment.
5. Cycle Slip Detection and Anomaly Investigation.
6. Satellite Arc Segmentation and Continuity Analysis.
7. Relative STEC Processing and Validation.
8. Calibrated STEC Processing and Bias Correction.
9. VTEC Conversion and Delta TEC Analysis.
10. Geomagnetic Storm Response Analysis.
11. Comparative Analysis of Geomagnetic Storm Events.

Each stage must be evaluated before its output is used by the next stage.

## 3. Project Directory Structure

Pipeline_Final/
- data/
  - raw/
    - DOY300/
    - DOY301/
    - DOY302/
    - DOY303/
    - DOY304/
    - DOY305/
    - DOY306/
    - DOY307/
  - external/
- documentations/
  - README_Pipeline_v0.1.md
- notebooks/
  - 00_Dataset_Inventory_and_Integrity_Audit.ipynb
  - 01_RINEX_Data_Ingestion_and_Parser_Development.ipynb
  - 02_RINEX_Parser_Validation_and_Data_Reconciliation.ipynb
- Output/
- config.py
- processing_functions.py
- rinex_parser.py

Existing Python modules are retained but have not yet been approved for reuse in the new pipeline.

## 4. Dataset Inventory and Initial Audit

Eight local RINEX files were identified in the raw data directory.

The initial audit examined:

- File and folder identity.
- Observation date recorded in the RINEX header.
- First and last observation epochs.
- Nominal temporal sampling interval.
- Daily 30-second grid coverage.
- Missing epochs at the beginning, middle, and end of the day.
- Unmapped and duplicate grid epochs.

### 4.1 Initial Audit Results

| DOY | Observation Date | Observed Epochs | Coverage (%) | Initial Status |
|---|---|---:|---:|---|
| 300 | 2003-10-27 | 2878 | 99.930556 | ELIGIBLE_FOR_NEXT_STAGE |
| 301 | 2003-10-28 | 2878 | 99.930556 | ELIGIBLE_FOR_NEXT_STAGE |
| 302 | 2003-10-29 | 2878 | 99.930556 | ELIGIBLE_FOR_NEXT_STAGE |
| 303 | 2003-10-30 | 2878 | 99.930556 | ELIGIBLE_FOR_NEXT_STAGE |
| 304 | 2003-10-31 | 2867 | 99.548611 | REVIEW_BOUNDARY_COVERAGE |
| 305 | 2003-11-08 | 2878 | 99.930556 | HOLD_IDENTITY_MISMATCH |
| 306 | 2003-11-02 | 2878 | 99.930556 | ELIGIBLE_FOR_NEXT_STAGE |
| 307 | 2003-11-03 | 2878 | 99.930556 | ELIGIBLE_FOR_NEXT_STAGE |

### 4.2 Important Findings

**DOY300–303 and DOY306–307**

Each file contains 2,878 observed epochs against 2,880 expected daily epochs. One epoch is missing at the beginning and one at the end of the daily grid. No internal missing grid slots were identified.

**DOY304**

The file contains 2,867 observed epochs. Twelve grid slots are missing at the beginning and one at the end of the day. No internal missing grid slots were identified. This file is retained for further review.

**DOY305**

The file is stored in the DOY305 directory, but its RINEX header indicates an observation date of 8 November 2003 rather than 1 November 2003.

The source of this discrepancy has not yet been established. The file is retained in the raw directory but placed on hold.

## 5. Audit Outputs

The initial audit generates the following files in Output/:

- dataset_inventory.csv
- dataset_header_identity_audit.csv
- dataset_temporal_audit.csv
- dataset_grid_coverage_audit.csv
- dataset_initial_eligibility.csv
- dataset_final_audit_report.csv
- dataset_final_audit_report.txt

## 6. Current Pipeline Status

### Notebook 00 — Dataset Inventory and Integrity Audit

**Status:** Initial audit completed.

The notebook documents the available raw files, header identity, temporal coverage, and initial eligibility classification.

ELIGIBLE_FOR_NEXT_STAGE does not mean that all GNSS observations have passed quality control. Observation-level validation remains necessary.

### Notebook 01 — RINEX Data Ingestion and Parser Development

**Status:** Planned / not yet validated.

Objectives:

- Read RINEX 2.10 header metadata.
- Identify observation types and their ordering.
- Develop a manual parser.
- Handle epoch records, satellite lists, and observation records.
- Support continuation records and event flags.
- Generate a structured observation table.

### Notebook 02 — RINEX Parser Validation and Data Reconciliation

**Status:** Planned / not yet validated.

Objectives:

- Compare manual parser output with raw RINEX records.
- Compare parsed observations with GeoRinex output.
- Reconcile timestamps, satellite identifiers, and observation values.
- Investigate discrepancies.
- Establish parser pass/fail criteria.

The manual parser will only become the official data source after validation requirements have been satisfied.

## 7. Planned Processing Stages

| Notebook | Processing Stage |
|---|---|
| 03 | GNSS Observation Quality Assessment |
| 04 | Cycle Slip Detection and Anomaly Investigation |
| 05 | Satellite Arc Segmentation and Continuity Analysis |
| 06 | Relative STEC Processing and Validation |
| 07 | Calibrated STEC Processing and Bias Correction |
| 08 | VTEC Conversion and Delta TEC Analysis |
| 09 | Geomagnetic Storm Response Analysis |
| 10 | Comparative Analysis of Geomagnetic Storm Events |

These stages are a preliminary roadmap and may be revised after technical evaluation and supervisor discussion.

## 8. Methodological Notes and Limitations

- The RINEX observation time system must be identified and converted explicitly before comparison with UTC-based geomagnetic indices.
- Successful temporal grid mapping does not establish observation-level quality.
- Parser validation must account for RINEX record structure, including continuation records and event flags.
- Relative STEC is not equivalent to calibrated absolute STEC.
- VTEC processing requires appropriate satellite geometry and a mapping function.
- Navigation or precise orbit data availability must be verified before VTEC processing.
- DOY305 must not be treated as 1 November 2003 data until its identity discrepancy is resolved.
- Data eligibility and processing decisions must be documented to preserve reproducibility.

## 9. Version Information

**Pipeline Version:** v0.1

**Current Focus:** Notebook 00–02.

**Development Principle:** Build, test, validate, and document each processing stage before proceeding to the next stage.
"""

# 4. SAVE README FILE
README_PATH = DOCUMENTATION_DIR / "README_Pipeline_v0.1.md"

README_PATH.write_text(
    README_CONTENT,
    encoding="utf-8"
)

# 5. VERIFY OUTPUT
if README_PATH.exists():
    print("README CREATED SUCCESSFULLY")
    print("=" * 50)
    print(f"File       : {README_PATH.name}")
    print(f"Directory  : {DOCUMENTATION_DIR}")
    print(f"File size  : {README_PATH.stat().st_size:,} bytes")
else:
    raise RuntimeError("README creation failed.")

README CREATED SUCCESSFULLY
File       : README_Pipeline_v0.1.md
Directory  : C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\documentations
File size  : 7,021 bytes


In [28]:

from pathlib import Path

# ============================================================
# 1. MENENTUKAN LOKASI PROYEK
# ============================================================

PROJECT_DIR = Path(
    r"C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final"
)

DOCUMENTATION_DIR = PROJECT_DIR / "documentations"
OUTPUT_DIR = PROJECT_DIR / "Output"

# Memastikan direktori tersedia
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Folder proyek tidak ditemukan: {PROJECT_DIR}"
    )

if not DOCUMENTATION_DIR.exists():
    raise FileNotFoundError(
        f"Folder dokumentasi tidak ditemukan: {DOCUMENTATION_DIR}"
    )

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"Folder Output tidak ditemukan: {OUTPUT_DIR}"
    )


# ============================================================
# 2. MENYUSUN ISI README
# ============================================================

README_CONTENT = r"""
# DOKUMENTASI PIPELINE PENELITIAN GNSS-TEC

**Versi:** 0.1  
**Dataset utama:** Halloween Geomagnetic Storm 2003  
**Stasiun GNSS utama:** BAKO  
**Format data:** RINEX Observation 2.10  

---

## 1. Gambaran Umum Penelitian

Penelitian ini berfokus pada pengamatan respons ionosfer ekuator Indonesia
terhadap aktivitas badai geomagnetik menggunakan data Global Navigation
Satellite System (GNSS).

Parameter utama yang dikaji adalah Total Electron Content (TEC), yang
diperoleh melalui pengolahan data observasi GNSS.

Peristiwa badai geomagnetik Halloween 2003 digunakan sebagai salah satu
objek utama penelitian. Data observasi GNSS dari stasiun BAKO digunakan
sebagai dataset utama untuk tahap pengembangan dan validasi pipeline.

Penelitian ini juga direncanakan mencakup perbandingan dengan peristiwa
badai geomagnetik 10–11 Mei 2024, bergantung pada ketersediaan dan
kelayakan data observasi.

## 2. Tujuan Pengembangan Pipeline

Pipeline dikembangkan untuk menghasilkan alur pengolahan data yang
sistematis, terdokumentasi, dan dapat direproduksi.

Tahapan utama yang direncanakan meliputi:

1. Inventarisasi dan pemeriksaan integritas dataset.
2. Pembacaan struktur dan pengembangan parser RINEX.
3. Validasi parser terhadap data mentah dan hasil GeoRinex.
4. Pemeriksaan kualitas observasi GNSS.
5. Deteksi cycle slip dan investigasi anomali fase.
6. Segmentasi arc satelit.
7. Pengolahan Relative STEC.
8. Kalibrasi STEC dan koreksi bias.
9. Konversi STEC menjadi VTEC serta analisis perubahan TEC.
10. Analisis respons TEC terhadap fase badai geomagnetik.
11. Perbandingan respons ionosfer pada peristiwa badai geomagnetik.

Tahapan tersebut merupakan rancangan alur penelitian. Setiap tahap
dilaksanakan setelah tahap sebelumnya memenuhi kriteria pemeriksaan.

## 3. Struktur Direktori Proyek

Struktur direktori utama:

```text
Pipeline_Final/
│
├── data/
│   ├── raw/
│   │   ├── DOY300/
│   │   ├── DOY301/
│   │   ├── DOY302/
│   │   ├── DOY303/
│   │   ├── DOY304/
│   │   ├── DOY305/
│   │   ├── DOY306/
│   │   └── DOY307/
│   │
│   └── external/
│
├── notebooks/
│
├── Output/
│
├── documentations/
│
├── config.py
├── processing_functions.py
└── rinex_parser.py
```

Keterangan:

- `data/raw/` menyimpan data RINEX mentah.
- `data/external/` digunakan untuk menyimpan data pendukung dari sumber eksternal.
- `notebooks/` menyimpan notebook pengolahan dan analisis.
- `Output/` menyimpan hasil pemeriksaan, tabel, dan keluaran pengolahan.
- `documentations/` menyimpan dokumentasi pipeline dan catatan metodologi.
- `config.py`, `processing_functions.py`, dan `rinex_parser.py` merupakan berkas pendukung kode pipeline yang akan digunakan setelah diperiksa dan divalidasi.

## 4. Dataset Halloween Storm 2003

Dataset utama yang diinventarisasi mencakup delapan folder DOY300–DOY307.

| DOY | Tanggal target | Nama berkas |
|---|---|---|
| 300 | 27 Oktober 2003 | bako3000.03o |
| 301 | 28 Oktober 2003 | bako3010.03o |
| 302 | 29 Oktober 2003 | bako3020.03o |
| 303 | 30 Oktober 2003 | bako3030.03o |
| 304 | 31 Oktober 2003 | bako3040.03o |
| 305 | 1 November 2003 | bako3050.03o |
| 306 | 2 November 2003 | bako3060.03o |
| 307 | 3 November 2003 | bako3070.03o |

Seluruh delapan berkas ditemukan dalam direktori data mentah.

### 4.1 Metadata RINEX DOY300

Berdasarkan pemeriksaan header berkas `bako3000.03o`:

- Versi RINEX: 2.10.
- Jenis data: GPS Observation Data.
- Nama stasiun: BAKO.
- Nomor marker: 23101M002.
- Receiver: TRIMBLE 4000SSE.
- Antena: TRM14532.00.
- Jumlah tipe observasi: 6.
- Tipe observasi: L1, L2, C1, P2, P1, dan D1.
- Sistem waktu pada header: GPS.
- Waktu observasi pertama: 27 Oktober 2003 pukul 00:00:30 GPS.

Koordinat stasiun yang tercantum pada header:

- X = -1836967.8873 m
- Y = 6065615.4384 m
- Z = -716260.3127 m

## 5. Hasil Audit Dataset

Pemeriksaan awal dilakukan untuk mengevaluasi kesesuaian identitas berkas,
rentang waktu observasi, keteraturan epoch, dan cakupan data terhadap
grid interval 30 detik.

### 5.1 Audit Identitas Dataset

Hasil pemeriksaan header terhadap delapan berkas:

| Klasifikasi | Jumlah berkas |
|---|---:|
| MATCH | 7 |
| DATE_MISMATCH | 1 |
| Total | 8 |

Temuan penting:

Berkas DOY305 bernama `bako3050.03o` memiliki tanggal observasi pertama
pada 8 November 2003 pukul 00:00:30 GPS, bukan 1 November 2003 seperti
tanggal target folder.

Berkas tersebut diberi status:

`HOLD_IDENTITY_MISMATCH`

Berkas tetap dipertahankan dalam direktori data mentah. Penyebab
ketidaksesuaian tanggal belum dapat dipastikan.

### 5.2 Audit Temporal dan Cakupan Epoch

Pemeriksaan temporal menunjukkan:

- DOY300, DOY301, DOY302, DOY303, DOY306, dan DOY307 memiliki 2878 epoch.
- DOY304 memiliki 2867 epoch.
- DOY305 memiliki 2878 epoch, tetapi tanggal observasinya tidak sesuai
  dengan tanggal target folder.

Untuk DOY300, interval observasi nominal adalah 30 detik.

Hasil pemetaan ke grid 30 detik:

| Parameter | Hasil |
|---|---:|
| Epoch yang diharapkan | 2880 |
| Epoch teramati | 2878 |
| Epoch berhasil dipetakan | 2878 |
| Epoch tidak terpetakan | 0 |
| Epoch duplikat | 0 |
| Epoch hilang | 2 |
| Cakupan grid | 99,930556% |

Dua epoch yang hilang berada pada awal dan akhir rentang observasi:
00:00:00 dan 23:59:30.

Tidak ditemukan gap internal pada grid DOY300.

### 5.3 Klasifikasi Kelayakan Awal

Berdasarkan audit identitas dan temporal:

| Status | DOY | Jumlah |
|---|---|---:|
| ELIGIBLE_FOR_NEXT_STAGE | 300–303, 306–307 | 6 |
| REVIEW_BOUNDARY_COVERAGE | 304 | 1 |
| HOLD_IDENTITY_MISMATCH | 305 | 1 |

**Catatan:** Status ELIGIBLE_FOR_NEXT_STAGE hanya menunjukkan bahwa
berkas lolos kriteria pemeriksaan awal identitas dan cakupan temporal.
Status ini belum menyatakan seluruh observasi GNSS bebas dari anomali
atau telah memenuhi persyaratan pengolahan TEC.

## 6. Rencana dan Struktur Notebook

### Tahap yang telah dikerjakan

**Notebook 00 — Dataset Inventory and Integrity Audit**

Fokus:
- Inventarisasi berkas RINEX.
- Pemeriksaan identitas stasiun dan tanggal observasi.
- Pemeriksaan header.
- Pemeriksaan rentang waktu dan interval epoch.
- Pemeriksaan cakupan grid 30 detik.
- Klasifikasi kelayakan awal dataset.

Keluaran utama:
- `doy300_integrity_audit_summary.csv`
- `dataset_header_identity_audit.csv`
- `dataset_temporal_audit.csv`
- `dataset_grid_coverage_audit.csv`
- `dataset_initial_eligibility.csv`
- `dataset_final_audit_report.csv`
- `dataset_final_audit_report.txt`

### Tahap berikutnya

**Notebook 01 — RINEX Data Ingestion and Parser Development**

Fokus:
- Memahami struktur RINEX 2.10.
- Membaca header dan urutan tipe observasi.
- Mengembangkan parser manual.
- Menangani struktur epoch, daftar satelit, dan continuation records.
- Menghasilkan tabel observasi terstruktur.

**Notebook 02 — RINEX Parser Validation and Data Reconciliation**

Fokus:
- Membandingkan hasil parser manual dengan data mentah.
- Membandingkan hasil parser dengan GeoRinex.
- Memeriksa jumlah satelit dan observasi.
- Memeriksa konsistensi timestamp.
- Menginvestigasi perbedaan hasil parsing, termasuk kasus G02 dan G14.
- Menetapkan kelayakan parser sebagai sumber data resmi pipeline.

Parser belum ditetapkan sebagai parser resmi sebelum proses validasi
pada Notebook 02 selesai.

### Roadmap tahap lanjutan

| Notebook | Nama | Fokus |
|---|---|---|
| 03 | GNSS Observation Quality Assessment | Kualitas dan kelengkapan observasi |
| 04 | Cycle Slip Detection and Anomaly Investigation | Deteksi cycle slip dan investigasi anomali |
| 05 | Satellite Arc Segmentation and Continuity Analysis | Segmentasi dan kontinuitas arc |
| 06 | Relative STEC Processing and Validation | Pengolahan dan validasi Relative STEC |
| 07 | Calibrated STEC Processing and Bias Correction | Kalibrasi STEC dan koreksi bias |
| 08 | VTEC Conversion and Delta TEC Analysis | Konversi VTEC dan analisis perubahan TEC |
| 09 | Geomagnetic Storm Response Analysis | Respons TEC terhadap badai geomagnetik |
| 10 | Comparative Analysis of Geomagnetic Storm Events | Perbandingan antarperistiwa badai |

Notebook 03–10 masih merupakan roadmap dan belum dinyatakan selesai.
Pelaksanaannya bergantung pada hasil validasi tahap sebelumnya serta
ketersediaan data pendukung.

## 7. Catatan Metodologi Penting

Beberapa hal yang harus diperhatikan dalam pengolahan selanjutnya:

1. Waktu pada header RINEX DOY300 dinyatakan dalam GPS Time. Konversi
   ke UTC harus dilakukan secara eksplisit sebelum dibandingkan dengan
   indeks geomagnetik atau data lain yang menggunakan UTC.

2. Struktur observasi RINEX 2.10 harus dibaca berdasarkan jumlah dan
   urutan tipe observasi pada header. Parser tidak boleh mengasumsikan
   bahwa setiap satelit selalu direpresentasikan oleh jumlah baris fisik
   yang sama tanpa memeriksa struktur berkas.

3. Loss-of-Lock Indicator (LLI) perlu diinterpretasikan berdasarkan bit
   penyusunnya. Nilai LLI tidak boleh langsung dianggap sebagai bukti
   tunggal terjadinya cycle slip.

4. Relative STEC yang dihitung secara independen pada setiap arc belum
   dapat diperlakukan sebagai STEC absolut yang kontinu.

5. Perhitungan STEC terkalibrasi memerlukan penanganan ambiguitas fase,
   penggabungan pengamatan fase dan pseudorange, serta koreksi bias yang
   relevan.

6. Konversi STEC menjadi VTEC memerlukan informasi geometri satelit,
   posisi IPP, elevasi satelit, dan fungsi pemetaan yang sesuai.

7. Perbandingan TEC dengan indeks geomagnetik harus menggunakan
   referensi waktu yang konsisten dan mempertimbangkan perbedaan
   resolusi temporal data.

8. Data pendukung seperti navigasi GNSS, orbit presisi, indeks
   geomagnetik, dan data ionosfer harus diperiksa ketersediaannya
   sebelum digunakan dalam tahapan analisis.

## 8. Prinsip Pengembangan Pipeline

Pipeline dikembangkan dengan prinsip:

- Setiap tahap memiliki tujuan dan keluaran yang jelas.
- Pemeriksaan kualitas dilakukan sebelum pengolahan fisik lebih lanjut.
- Hasil pengolahan harus dapat ditelusuri ke data mentah.
- Perubahan kode dan metode harus terdokumentasi.
- Asumsi dan keterbatasan data harus dinyatakan secara eksplisit.
- Tahap berikutnya tidak dilanjutkan sebelum kriteria validasi tahap
  sebelumnya terpenuhi.

## 9. Status Pengembangan

**Versi pipeline:** 0.1

**Status Notebook 00:** Audit awal dataset telah diselesaikan.

**Status Notebook 01:** Belum dimulai.

**Status Notebook 02:** Belum dimulai.

Pekerjaan selanjutnya difokuskan pada pengembangan dan validasi parser
RINEX sebelum melanjutkan ke pemeriksaan kualitas observasi dan
pengolahan TEC.

---

**Dokumen ini merupakan dokumentasi pengembangan pipeline versi 0.1.**
Pembaruan dilakukan mengikuti hasil pemeriksaan, validasi, dan keputusan
metodologi pada tahapan penelitian berikutnya.
"""


# ============================================================
# 3. MENYIMPAN README DENGAN NAMA BARU
# ============================================================

README_PATH = DOCUMENTATION_DIR / "README_Pipeline_v0.1_ID.md"

# Mencegah file yang sudah ada tertimpa
if README_PATH.exists():
    raise FileExistsError(
        f"File sudah ada dan tidak akan ditimpa: {README_PATH}\n"
        "Gunakan nama file baru jika ingin membuat salinan lain."
    )

with README_PATH.open("x", encoding="utf-8") as file:
    file.write(README_CONTENT.strip() + "\n")


# ============================================================
# 4. VERIFIKASI FILE
# ============================================================

print("README berhasil dibuat.")
print(f"Lokasi: {README_PATH}")
print(f"Ukuran: {README_PATH.stat().st_size:,} bytes")
print(f"File tersedia: {README_PATH.exists()}")


FileExistsError: File sudah ada dan tidak akan ditimpa: C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\documentations\README_Pipeline_v0.1_ID.md
Gunakan nama file baru jika ingin membuat salinan lain.

In [29]:

# ============================================================
# 3. MENYIMPAN README (AMAN UNTUK DIJALANKAN ULANG)
# ============================================================

README_PATH = DOCUMENTATION_DIR / "README_Pipeline_v0.1_ID.md"

README_PATH.write_text(
    README_CONTENT.strip() + "\n",
    encoding="utf-8"
)

print("README berhasil dibuat atau diperbarui.")
print(f"Lokasi: {README_PATH}")


README berhasil dibuat atau diperbarui.
Lokasi: C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\documentations\README_Pipeline_v0.1_ID.md
